# TerraTree (Kali track) — Step 1: Data Acquisition
Study area: **Kali Tiger Reserve, Karnataka, India**

This track uses REAL field ground-truth (56 vegetation plots, GBIF/Darwin
Core format, surveyed Nov-Dec 2025) instead of proxy reference datasets —
this is the genuine species/family-level classification track.

This notebook:
1. Authenticates Earth Engine
2. Loads the Kali Tiger Reserve boundary (OSM/Nominatim)
3. Pulls multi-temporal Sentinel-2 and Sentinel-1 composites
4. Visualizes everything, including your 56 real ground-truth plot locations
5. Exports composites to Drive (backup/record; notebook 02 rebuilds fresh from EE directly)

In [ ]:
!pip install geemap -q

In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='terratree')

## Define the study area

In [ ]:
import requests

QUERY_CANDIDATES = [
    "Kali Tiger Reserve, Karnataka, India",
    "Anshi Dandeli Tiger Reserve",
    "Dandeli Wildlife Sanctuary",
    "Kali Tiger Reserve",
]

aoi = None
for query in QUERY_CANDIDATES:
    resp = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": query, "format": "geojson", "polygon_geojson": 1, "limit": 1},
        headers={"User-Agent": "terratree-major-project"}
    )
    features = resp.json().get('features', [])
    if features:
        aoi = ee.Geometry(features[0]['geometry'])
        print(f"AOI matched on: '{query}'")
        break

if aoi is None:
    print("All OSM queries failed — using fallback bounding box based on ground-truth point spread.")
    aoi = ee.Geometry.Rectangle([74.20, 14.80, 74.75, 15.55])

## Load your ground-truth plots and overlay them

This is the key sanity check for this track: your 56 real field plots
should fall INSIDE (or very near) the AOI boundary. If they're clearly
outside, the OSM boundary matched the wrong place and needs fixing before
anything else.

In [ ]:
from google.colab import files
import pandas as pd

# Upload KTR_actual_ground_truth.csv when prompted
uploaded = files.upload()
gt_df = pd.read_csv('KTR_actual_ground_truth.csv')
print(gt_df.shape)
gt_df[['plot_id', 'decimalLatitude', 'decimalLongitude']].drop_duplicates().head()

In [ ]:
plot_locations = gt_df[['plot_id', 'decimalLatitude', 'decimalLongitude']].drop_duplicates()

Map = geemap.Map(basemap='HYBRID')
Map.centerObject(aoi, 10)
Map.addLayer(aoi, {'color': 'red'}, 'Study area (Kali Tiger Reserve)')

for _, row in plot_locations.iterrows():
    Map.add_marker(location=[row['decimalLatitude'], row['decimalLongitude']])

Map

## Sentinel-2 optical collection

Date range centered around the survey period (Nov-Dec 2025) but widened
to get enough cloud-free scenes for a stable median composite — the
underlying forest canopy doesn't change fast enough for a slightly wider
window to meaningfully mismatch the ground survey.

In [ ]:
START_DATE = '2023-01-01'
END_DATE = '2025-12-31'
MAX_CLOUD_PCT = 30

def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask).divide(10000).copyProperties(image, ['system:time_start'])

s2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_PCT))
    .map(mask_s2_clouds)
)
print('Sentinel-2 scenes:', s2_collection.size().getInfo())

s2_median = s2_collection.median().clip(aoi)
NEEDED_BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
s2_median = s2_median.select(NEEDED_BANDS)  # trimmed from the start this time

Map.addLayer(s2_median, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'Sentinel-2 true color')
Map

## Sentinel-1 SAR collection

In [ ]:
s1_collection = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
)
print('Sentinel-1 scenes:', s1_collection.size().getInfo())

s1_median = s1_collection.select(['VV', 'VH']).median().toFloat().clip(aoi)
Map.addLayer(s1_median, {'bands': ['VV'], 'min': -20, 'max': 0}, 'Sentinel-1 VV')
Map

## Export composites to Drive (backup copies)

Notebook 02 rebuilds these fresh from Earth Engine directly rather than
reading these files back — same pattern as the Sundarbans track. These
exports are your record/backup copies.

In [ ]:
export_s2 = ee.batch.Export.image.toDrive(
    image=s2_median, description='kali_s2_median', folder='terratree',
    fileNamePrefix='kali_s2_median', region=aoi, scale=10, maxPixels=1e13
)
export_s2.start()

export_s1 = ee.batch.Export.image.toDrive(
    image=s1_median, description='kali_s1_median', folder='terratree',
    fileNamePrefix='kali_s1_median', region=aoi, scale=10, maxPixels=1e13
)
export_s1.start()

print("Exports started.")

import time
tasks = [export_s2, export_s1]
done_states = {'COMPLETED', 'FAILED', 'CANCELLED'}
while True:
    states = [t.status()['state'] for t in tasks]
    print(states)
    if all(s in done_states for s in states):
        break
    time.sleep(30)

## Next steps
- [ ] Confirm the ground-truth plot markers actually fall inside/near the AOI boundary on the map
- [ ] Confirm S2/S1 scene counts are reasonable (non-zero)
- [ ] Move to `02_feature_extraction_and_labels.ipynb` — this builds the real training table from your 56 ground-truth plots